# Session 08: RFM customer segmentation

Practice companion for New. **Synthetic demonstration data**, not the lecturer's original dataset. Run the code cells top to bottom. The original slide snippets remain available separately in the website.

Requires numpy, pandas, matplotlib, seaborn, scikit-learn, and plotly. Install missing packages with `%pip install numpy pandas matplotlib seaborn scikit-learn plotly`.

In [ ]:
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)
n = 600
df = pd.DataFrame({
    'order_id': np.arange(1, n+1),
    'customer_id': rng.integers(1, 81, n),
    'date': pd.Timestamp('2025-01-01') + pd.to_timedelta(rng.integers(0, 730, n), unit='D'),
    'product': rng.choice(['Notebook', 'Pen', 'Bag'], n),
    'region': rng.choice(['North', 'South'], n),
    'price': rng.uniform(50, 500, n).round(2),
    'quantity': rng.integers(1, 6, n),
})
df['category'] = df['product'].map({'Notebook': 'Stationery', 'Pen': 'Stationery', 'Bag': 'Accessories'})
df['revenue'] = df['price'] * df['quantity']
df['cost'] = df['revenue'] * rng.uniform(0.45, 0.85, n)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.to_period('M').astype(str)
snapshot = df['date'].max() + pd.Timedelta(days=1)
rfm = df.groupby('customer_id').agg(
    recency=('date', lambda s: (snapshot - s.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('revenue', 'sum'))
# Fixed business thresholds avoid arbitrary tie-breaking; these are illustrative.
rfm['segment'] = np.select(
    [(rfm.recency <= 60) & (rfm.frequency >= 8), rfm.recency > 180],
    ['Champions', 'At risk'], default='Developing')
# Slide-style quantile scores. Small or tied datasets need an explicit policy.
for field, label, labels in [('recency','R',[4,3,2,1]), ('frequency','F',[1,2,3,4]), ('monetary','M',[1,2,3,4])]:
    values = rfm[field]
    try:
        rfm[label] = pd.qcut(values, 4, labels=labels)
    except ValueError:
        # Demonstration fallback only: first-rank splits ties by row order.
        print(f'{field}: tied boundaries; applying documented rank-first fallback')
        rfm[label] = pd.qcut(values.rank(method='first'), 4, labels=labels)
print(rfm.head())
print(rfm.groupby('segment').agg(customers=('recency','size'),
      recency=('recency','mean'), frequency=('frequency','mean'), monetary=('monetary','mean')))


## Your turn
Build one RFM row per customer, assign transparent segment rules, profile the groups, and propose one action for each.

## Watch out
The slide uses count(order_id), which counts rows. Use nunique if transactions contain multiple lines per order. Arbitrary tie-breaking for qcut can split identical customers; document the choice.